# Outlier98 Model v3 Data Analysis

This notebook reproduces the public synthetic-data analysis for Model v3, which estimates the risk of low execution on the next day.

The dataset is entirely synthetic and contains no participant information.

All performance values are based on held-out synthetic data. They show how the pipeline behaves under the generator assumptions, but do not establish real-world validity, causal effects or population-level generalisability.

## 1. Analysis objective

The target is `next_day_low_execution`. It is positive when completed eligible planned execution on prediction day D is below 50%. Features are based on the seven complete preceding local days, D−7 to D−1. Prediction-day behavioural outcomes are excluded from the feature vector.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score,
                             f1_score, log_loss, brier_score_loss, confusion_matrix, roc_curve)

pd.set_option('display.max_columns', None)

# Works from the notebook directory, repository root, or a Colab checkout.
candidates = [
    Path('../data/model_v3_synthetic_dataset.csv'),
    Path('data/model_v3_synthetic_dataset.csv'),
    Path('/content/final_outlier98_thesis/data/model_v3_synthetic_dataset.csv'),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Place the repository in the working directory or update data_path.')

df = pd.read_csv(data_path)
print(f'Loaded: {data_path}')
print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

## 2. Dataset checks

The verified experiment contains 31,800 labelled observations from 600 synthetic users. The split is by synthetic user, not by row.

In [ ]:
assert len(df) == 31800
assert df['synthetic_user_id'].nunique() == 600
assert (df['split'] == 'train').sum() == 25440
assert (df['split'] == 'held_out').sum() == 6360
assert df.loc[df['split'] == 'train', 'synthetic_user_id'].nunique() == 480
assert df.loc[df['split'] == 'held_out', 'synthetic_user_id'].nunique() == 120

print(df.groupby('split').agg(rows=('synthetic_user_id', 'size'), users=('synthetic_user_id', 'nunique')))
print('\nMissing values by column:')
print(df.isna().sum())

## 3. Target distribution

The target is 1 for low execution and 0 otherwise. The majority-class baseline is the larger of the positive and negative class proportions.

In [ ]:
held = df[df['split'] == 'held_out'].copy()
target_rate = held['next_day_low_execution'].mean()
majority_baseline = max(target_rate, 1 - target_rate)
print(f'Held-out positive rate: {target_rate:.6f}')
print(f'Held-out majority-class baseline: {majority_baseline:.6f}')
assert round(majority_baseline, 3) == 0.610
held['next_day_low_execution'].value_counts().sort_index().plot(kind='bar', rot=0, title='Held-out target distribution')
plt.xlabel('next_day_low_execution')
plt.ylabel('Observations')
plt.show()

The held-out set contains more non-low-execution than low-execution observations, which is why the majority-class baseline is approximately 0.610.

## 4. Feature overview

The model uses nine features. `habit_completion_rate_7d` uses Done Habit-days in its numerator; Partial is not equivalent to Done for this feature. Partial contributes 0.5 only in target construction. Recorded focus duration is used rather than planned T-Block duration. Missing values are not behavioural zeros.

In [ ]:
features = [
    'habit_completion_rate_7d', 'task_completion_rate_7d', 'tblock_completion_rate_7d',
    'avg_focus_mins_7d', 'current_streak', 'days_since_last_active',
    'reminder_usage_rate', 'avg_mood_7d', 'is_weekend'
]
display(df[features].describe().T)
display(df[features].isna().sum().rename('missing_values').to_frame())
df[features].hist(figsize=(14, 10), bins=25)
plt.tight_layout()
plt.show()

Missing values are handled using the training-set means rather than being interpreted as behavioural zeros.

## 5. Train / held-out split

Splitting by synthetic user avoids leakage from overlapping behavioural windows of the same synthetic user. No synthetic user should appear in both subsets.

In [ ]:
train = df[df['split'] == 'train'].copy()
train_users = set(train['synthetic_user_id'])
held_users = set(held['synthetic_user_id'])
assert train_users.isdisjoint(held_users)
print(f'Training users: {len(train_users)}; held-out users: {len(held_users)}')
print(f'Training rows: {len(train)}; held-out rows: {len(held)}')

## 6. Preprocessing

Training-set means and population standard deviations are used for standardisation. Missing numerical values are replaced by frozen training means. Prediction-day outcomes are excluded from historical features.

In [ ]:
means = train[features].mean(skipna=True)
scales = train[features].std(ddof=0, skipna=True).replace(0, 1)
X_train = ((train[features].fillna(means) - means) / scales).to_numpy(dtype=float)
X_held = ((held[features].fillna(means) - means) / scales).to_numpy(dtype=float)
y_train = train['next_day_low_execution'].to_numpy(dtype=float)
y_held = held['next_day_low_execution'].to_numpy(dtype=int)
display(pd.DataFrame({'mean': means, 'population_std': scales}))

## 7. Model v3

Model v3 uses binary Logistic Regression. It is trained with full-batch gradient descent on L2-regularised log loss, using a learning rate of 0.5, 1,500 iterations, L2 regularisation of 0.001 and seed 20261007. For this public notebook, the calculation is reproduced directly in NumPy so that the analysis can run independently of the private application code.

In [ ]:
learning_rate = 0.5
iterations = 1500
l2 = 0.001
seed = 20261007

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

weights = np.zeros(X_train.shape[1], dtype=float)
intercept = 0.0
for _ in range(iterations):
    probabilities = sigmoid(intercept + X_train @ weights)
    error = probabilities - y_train
    gradient_w = (X_train.T @ error) / len(X_train) + l2 * weights
    gradient_b = error.mean()
    weights -= learning_rate * gradient_w
    intercept -= learning_rate * gradient_b

print('Intercept:', round(intercept, 10))
display(pd.DataFrame({'feature': features, 'coefficient': weights}))

## 8. Held-out predictions and classification metrics

In [ ]:
held_probability = sigmoid(intercept + X_held @ weights)
held_prediction = (held_probability >= 0.5).astype(int)
metrics = {
    'ROC-AUC': roc_auc_score(y_held, held_probability),
    'Accuracy': accuracy_score(y_held, held_prediction),
    'Precision': precision_score(y_held, held_prediction),
    'Recall': recall_score(y_held, held_prediction),
    'F1': f1_score(y_held, held_prediction),
    'Log loss': log_loss(y_held, held_probability),
    'Brier score': brier_score_loss(y_held, held_probability),
}
metrics_table = pd.DataFrame.from_dict(metrics, orient='index', columns=['value'])
display(metrics_table)
assert round(metrics['ROC-AUC'], 3) == 0.878
assert round(metrics['Accuracy'], 3) == 0.803
assert round(metrics['Precision'], 3) == 0.760
assert round(metrics['Recall'], 3) == 0.722
assert round(metrics['F1'], 3) == 0.740
assert round(metrics['Log loss'], 3) == 0.433
assert round(metrics['Brier score'], 3) == 0.138

On held-out synthetic users, the model performs above the majority-class baseline and shows useful discrimination, although this result remains specific to the synthetic evaluation setting.

## 9. Confusion matrix

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_held, held_prediction).ravel()
confusion = pd.DataFrame(
    [[tp, fn], [fp, tn]],
    index=['Actual low execution', 'Actual not low execution'],
    columns=['Predicted low execution', 'Predicted not low execution']
)
display(confusion)
assert (tp, fp, tn, fn) == (1789, 566, 3316, 689)

The matrix shows that the model identifies many low-execution observations while still producing both false-positive and false-negative predictions.

## 10. ROC analysis

In [ ]:
fpr, tpr, _ = roc_curve(y_held, held_probability)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC-AUC = {metrics["ROC-AUC"]:.3f}')
plt.plot([0, 1], [0, 1], '--', color='grey', label='Chance')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('Held-out synthetic ROC curve')
plt.legend()
plt.show()

The ROC curve shows discrimination above chance for the held-out synthetic observations.

## 11. Calibration analysis

Risk bands are Low for probabilities below 0.40, Medium for 0.40 to below 0.70, and High for probabilities of 0.70 or above.

In [ ]:
band = pd.cut(held_probability, bins=[-np.inf, 0.4, 0.7, np.inf], labels=['low', 'medium', 'high'], right=False)
calibration = pd.DataFrame({'risk_band': band, 'probability': held_probability, 'outcome': y_held}).groupby('risk_band', observed=False).agg(
    samples=('outcome', 'size'),
    mean_predicted=('probability', 'mean'),
    observed_low_execution_rate=('outcome', 'mean')
)
display(calibration)
assert list(calibration['samples']) == [3674, 1138, 1548]
assert [round(value, 4) for value in calibration['mean_predicted']] == [0.1366, 0.5620, 0.8188]
assert [round(value, 4) for value in calibration['observed_low_execution_rate']] == [0.1418, 0.5791, 0.8385]
calibration[['mean_predicted', 'observed_low_execution_rate']].plot(kind='bar', rot=0, title='Held-out synthetic calibration by risk band')
plt.ylabel('Probability / observed rate')
plt.show()

Within the synthetic held-out set, the observed low-execution rates are close to the mean predicted probabilities across the three risk bands.

## 12. Model interpretation

For a non-imputed feature, the linear explanation contribution is `coefficient × standardised feature`. This describes how an input affects the fitted model score. SHAP is not used. Coefficient contributions are not causal effects.

## 13. Results summary

The following table summarises the reproduced held-out synthetic results.

In [ ]:
summary = pd.DataFrame({
    'metric': ['ROC-AUC', 'Accuracy', 'Precision', 'Recall', 'F1', 'Log loss', 'Brier score'],
    'value': [metrics['ROC-AUC'], metrics['Accuracy'], metrics['Precision'], metrics['Recall'], metrics['F1'], metrics['Log loss'], metrics['Brier score']]
})
display(summary)

## 14. Limitations

Training data and held-out evaluation data are synthetic. The synthetic generator introduces structural circularity because outcomes are partly generated through related behavioural assumptions. The results show technical pipeline behaviour, not real-world predictive validity, causality or population-level generalisability.

The exploratory three-user real-user evaluation is separate from this synthetic experiment and is not statistical validation of Model v3.